### RAG Pipeline --> DATA ingestion to Vector Db Pipeline

In [3]:
import os 
from langchain_community.document_loaders import PyMuPDFLoader,PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [4]:
# read all the pdf in the directory  
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  X Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

#
#  Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 1 PDF files to process

Processing: book.pdf
  ✓ Loaded 586 pages

Total documents loaded: 586


### Text splitting text into chunks

In [7]:
# text splitting gets into chunks 
def split_document(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance """
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n","",""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} document into {len(split_docs)} chunks")

    #show an example of chunks
    if split_docs:
        print(f"\n Example")
        print(f"content : {split_docs[0].page_content[:200]} ...")
        print(f"content : {split_docs[0].metadata} ...")
    return split_docs

In [8]:
chunks = split_document(all_pdf_documents)

split 586 document into 1536 chunks

 Example
content : 1 ...
content : {'producer': 'calibre (5.14.0) [https://calibre-ebook.com]', 'creator': 'calibre (5.14.0) [https://calibre-ebook.com]', 'creationdate': '2021-06-18T14:36:04+00:00', 'author': 'Lucius Seneca', 'keywords': 'stoicism, phylosophy, mindfulness', 'moddate': '2021-06-18T08:36:07-06:00', 'title': 'Letters From a Stoic', 'source': '..\\data\\pdf\\book.pdf', 'total_pages': 586, 'page': 1, 'page_label': '2', 'source_file': 'book.pdf', 'file_type': 'pdf'} ...


In [9]:
chunks

[Document(metadata={'producer': 'calibre (5.14.0) [https://calibre-ebook.com]', 'creator': 'calibre (5.14.0) [https://calibre-ebook.com]', 'creationdate': '2021-06-18T14:36:04+00:00', 'author': 'Lucius Seneca', 'keywords': 'stoicism, phylosophy, mindfulness', 'moddate': '2021-06-18T08:36:07-06:00', 'title': 'Letters From a Stoic', 'source': '..\\data\\pdf\\book.pdf', 'total_pages': 586, 'page': 1, 'page_label': '2', 'source_file': 'book.pdf', 'file_type': 'pdf'}, page_content='1'),
 Document(metadata={'producer': 'calibre (5.14.0) [https://calibre-ebook.com]', 'creator': 'calibre (5.14.0) [https://calibre-ebook.com]', 'creationdate': '2021-06-18T14:36:04+00:00', 'author': 'Lucius Seneca', 'keywords': 'stoicism, phylosophy, mindfulness', 'moddate': '2021-06-18T08:36:07-06:00', 'title': 'Letters From a Stoic', 'source': '..\\data\\pdf\\book.pdf', 'total_pages': 586, 'page': 2, 'page_label': '3', 'source_file': 'book.pdf', 'file_type': 'pdf'}, page_content='2\nThis eBook is the result of 